In [2]:
# 1.2
# 1. import the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp

# 2. import csv
data = pd.read_csv('./data_and_materials/gamma-ray.csv')
# check the data
data.head()


,seconds,count
0,116.0,0.0
1,112.0,0.0
2,160.0,0.0
3,51.5,0.0
4,102.0,1.0


## Problem 1.2, part (d)

Under the null model, the emission rate is constant:

$$G_i \sim \mathrm{Poisson}(\lambda t_i).$$

The likelihood is

$$L(\lambda)=\prod_i \frac{e^{-\lambda t_i}(\lambda t_i)^{G_i}}{G_i!}.$$

The log-likelihood, up to constants that do not depend on $\lambda$, is

$$\ell(\lambda)=-\lambda\sum_i t_i + \left(\sum_i G_i\right)\log \lambda + C.$$

Differentiate and set the derivative equal to zero:

$$\frac{d\ell}{d\lambda}=-\sum_i t_i + \frac{\sum_i G_i}{\lambda}=0.$$

Therefore the MLE is

$$\hat\lambda=\frac{\sum_i G_i}{\sum_i t_i}.$$


In [3]:
total_count = data['count'].sum()
total_time = data['seconds'].sum()
lambda_hat_null = total_count / total_time

print(f"sum G_i = {total_count:g}")
print(f"sum t_i = {total_time:g}")
print(f"lambda_hat = {lambda_hat_null}")
print(f"3 significant figures: {lambda_hat_null:.3g}")

sum G_i = 61
sum t_i = 15718.2
lambda_hat = 0.0038808514969907496
3 significant figures: 0.00388


For the answer box in part (d), enter

$$\boxed{0.00388}$$

This is a rate per second. It is not the total number of observed gamma rays.

## Problem 1.2, part (e)

Under the alternative model, each interval has its own rate:

$$G_i \sim \mathrm{Poisson}(\lambda_i t_i).$$

For one interval, the log-likelihood is

$$\ell_i(\lambda_i)=-\lambda_i t_i + G_i\log(\lambda_i t_i)-\log(G_i!).$$

Keeping only terms involving $\lambda_i$:

$$\ell_i(\lambda_i)=-\lambda_i t_i + G_i\log\lambda_i + C.$$

Differentiate and set equal to zero:

$$\frac{d\ell_i}{d\lambda_i}=-t_i+\frac{G_i}{\lambda_i}=0.$$

So the MLE for each interval is

$$\hat\lambda_i=\frac{G_i}{t_i}.$$

For the answer box in part (e), enter `G_i/t_i`.

In [ ]:
data['lambda_hat_i'] = data['count'] / data['seconds']
data.head()

## Problem 1.4

这一题的目标是：对 3051 个基因分别做一次 ALL vs AML 的两样本 Welch t-test，然后统计在显著性水平 $\alpha=0.05$ 下有多少个基因显著。

对第 $i$ 个基因，原假设和备择假设是

$$H_{0,i}: \mu_{\mathrm{ALL},i}=\mu_{\mathrm{AML},i}, \qquad H_{1,i}: \mu_{\mathrm{ALL},i}\ne\mu_{\mathrm{AML},i}.$$

因为题目给出的统计量是 Welch unequal variances t-test，所以不假设 ALL 和 AML 两组方差相同。每个基因都会得到一个双侧 p-value。Part (a) 直接数未校正 p-value 不超过 0.05 的基因数；Part (b) 先对 3051 个 p-value 做多重检验校正，再数校正后显著的基因数。

In [1]:
# Problem 1.4: load Golub gene expression data
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

golub_dir = Path('data_and_materials/golub_data')
golub = pd.read_csv(golub_dir / 'golub.csv', index_col=0)
golub_class = pd.read_csv(golub_dir / 'golub_cl.csv', index_col=0)['x']

all_samples = golub.loc[:, golub_class.to_numpy() == 0]
aml_samples = golub.loc[:, golub_class.to_numpy() == 1]

print('golub shape:', golub.shape)
print('ALL samples:', all_samples.shape[1])
print('AML samples:', aml_samples.shape[1])

golub shape: (3051, 38)
ALL samples: 27
AML samples: 11


### Welch t-test 的计算

对每个基因，先计算两组样本均值、样本方差和标准误：

$$SE_i=\sqrt{\frac{s^2_{\mathrm{ALL},i}}{N_{\mathrm{ALL}}}+\frac{s^2_{\mathrm{AML},i}}{N_{\mathrm{AML}}}}.$$

Welch 统计量是

$$t_i=\frac{\bar X_{\mathrm{ALL},i}-\bar X_{\mathrm{AML},i}}{SE_i}.$$

自由度使用题目给出的 Welch-Satterthwaite 近似。因为题目问表达水平是否不同，所以使用双侧 p-value：

$$p_i=2P(T_{\nu_i}\ge |t_i|).$$

In [2]:
# Manual Welch t-test, following the formulas in the problem statement
n_all = all_samples.shape[1]
n_aml = aml_samples.shape[1]

mean_all = all_samples.mean(axis=1)
mean_aml = aml_samples.mean(axis=1)
var_all = all_samples.var(axis=1, ddof=1)
var_aml = aml_samples.var(axis=1, ddof=1)

se = np.sqrt(var_all / n_all + var_aml / n_aml)
t_welch = (mean_all - mean_aml) / se

df_welch = (var_all / n_all + var_aml / n_aml) ** 2 / (
    (var_all / n_all) ** 2 / (n_all - 1)
    + (var_aml / n_aml) ** 2 / (n_aml - 1)
)

p_values = 2 * stats.t.sf(np.abs(t_welch), df_welch)

# This should match scipy.stats.ttest_ind(..., equal_var=False).
scipy_p_values = stats.ttest_ind(
    all_samples,
    aml_samples,
    axis=1,
    equal_var=False,
    nan_policy='omit'
).pvalue

print('max absolute difference from scipy:', np.max(np.abs(p_values - scipy_p_values)))

max absolute difference from scipy: 8.881784197001252e-16


### Parts (a) and (b)

未校正时，直接计算 $p_i\le 0.05$ 的个数。

Holm-Bonferroni 是控制 FWER 的 step-down 方法，通常比普通 Bonferroni 略不保守。Benjamini-Hochberg 是控制 FDR 的方法，通常会比 Holm-Bonferroni 给出更多拒绝。这里用 `statsmodels.stats.multitest.multipletests` 对同一批 p-value 做校正。

In [3]:
alpha = 0.05

num_uncorrected = int((p_values <= alpha).sum())

reject_holm, p_holm, _, _ = multipletests(p_values, alpha=alpha, method='holm')
reject_bh, p_bh, _, _ = multipletests(p_values, alpha=alpha, method='fdr_bh')

num_holm = int(reject_holm.sum())
num_bh = int(reject_bh.sum())

print('Part (a), uncorrected p-values:', num_uncorrected)
print('Part (b), Holm-Bonferroni:', num_holm)
print('Part (b), Benjamini-Hochberg:', num_bh)

Part (a), uncorrected p-values: 1078
Part (b), Holm-Bonferroni: 103
Part (b), Benjamini-Hochberg: 695


所以答题框中应填写：

- Part (a), uncorrected p-values: $\boxed{1078}$
- Part (b), Holm-Bonferroni: $\boxed{103}$
- Part (b), Benjamini-Hochberg: $\boxed{695}$

### Part (c)

答案是 **Yes**。

把 p-value 从小到大排序为 $p_{(1)}\le\cdots\le p_{(m)}$。Holm-Bonferroni 在第 $k$ 个位置使用的临界值是

$$\frac{\alpha}{m-k+1},$$

而 Benjamini-Hochberg 在第 $k$ 个位置使用的临界值是

$$\frac{k\alpha}{m}.$$

对所有 $k=1,\ldots,m$，都有

$$\frac{\alpha}{m-k+1}\le \frac{k\alpha}{m},$$

因为这等价于 $m\le k(m-k+1)$，也就是 $(k-1)(m-k)\ge0$。因此只要某个排序位置能通过 Holm-Bonferroni 的阈值，它也不会超过 Benjamini-Hochberg 对应位置的阈值。所以在同一批数据、同一个 $\alpha$ 下，BH 的拒绝数总是至少和 Holm-Bonferroni 一样多。

## Problem 1.6

这一题围绕 ordinary least squares (OLS) 和 gradient descent。下面的代码都直接从 `data_and_materials` 文件夹读取数据，并且在关键步骤加了注释，方便之后复查每一步在做什么。

### Part (a)

OLS 要最小化

$$(y-X\beta)^T(y-X\beta).$$

对 $\beta$ 求导并令导数为 0：

$$-2X^T(y-X\beta)=0 \Rightarrow X^TX\beta=X^Ty.$$

所以矩阵形式的 OLS 估计量是

$$\boxed{\hat\beta=(X^TX)^{-1}X^Ty}.$$

因此 Part (a) 选择第一项。

In [ ]:
# Problem 1.6 setup: load the synthetic design matrix X and response vector y.
# These CSV files have no header row, so np.genfromtxt is the simplest loader.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

data_dir = Path('data_and_materials')

syn_X = np.genfromtxt(data_dir / 'syn_X.csv', delimiter=',')
syn_y = np.genfromtxt(data_dir / 'syn_y.csv', delimiter=',')

def add_intercept(X):
    """Add a leading column of 1s so beta[0] is the intercept."""
    return np.concatenate((np.ones_like(X[:, :1]), X), axis=1)

syn_X_with_intercept = add_intercept(syn_X)

print('syn_X shape before intercept:', syn_X.shape)
print('syn_X shape after intercept:', syn_X_with_intercept.shape)
print('syn_y shape:', syn_y.shape)

### Part (b)

这里要求把 Part (a) 的公式应用到合成数据上。注意题目要求添加截距项，所以先在 $X$ 的最前面加一列 1。

In [ ]:
# Closed-form OLS estimator: beta_hat = (X^T X)^(-1) X^T y.
# np.linalg.solve is numerically cleaner than explicitly computing the inverse,
# but it is solving the same normal equation X^T X beta = X^T y.
XtX = syn_X_with_intercept.T @ syn_X_with_intercept
Xty = syn_X_with_intercept.T @ syn_y
beta_hat_syn = np.linalg.solve(XtX, Xty)

print('OLS beta_hat:', beta_hat_syn)
print('Three significant figures:', [float(f'{b:.3g}') for b in beta_hat_syn])

Part (b) 的三位有效数字答案是

$$\boxed{[1.93,\ 1.26,\ -4.60]}.$$

### Part (c)

目标函数是

$$L(\beta)=(y-X\beta)^T(y-X\beta).$$

它的梯度是

$$\nabla_\beta L=2X^T(X\beta-y).$$

下面实现 batch gradient descent。因为我们用了完整梯度 `2 * X.T @ (X @ beta - y)`，固定步长收敛需要大致满足

$$0<\alpha<\frac{1}{\lambda_{\max}(X^TX)}.$$

对二次目标函数，接近最优的固定步长通常在

$$\alpha\approx\frac{1}{\lambda_{\min}(X^TX)+\lambda_{\max}(X^TX)}$$

附近。

In [ ]:
def gradient_descent(X, y, step_size, precision=1e-6, max_iter=200_000):
    """Run batch gradient descent for least squares.

    Parameters
    ----------
    X, y : array-like
        Design matrix and response vector. X should already include the intercept
        column if an intercept is desired.
    step_size : float
        The gradient descent learning rate alpha.
    precision : float
        Stop when the relative change in beta is smaller than this value.
    max_iter : int
        Safety cap so a bad step size cannot run forever.
    """
    beta = np.zeros(X.shape[1])

    for iteration in range(1, max_iter + 1):
        # Gradient of (y - X beta)^T (y - X beta).
        gradient = 2 * X.T @ (X @ beta - y)
        beta_next = beta - step_size * gradient

        # If the step size is too large, beta can overflow or become NaN.
        if not np.all(np.isfinite(beta_next)):
            return beta_next, iteration, False

        # Relative stopping rule for beta.
        relative_change = np.linalg.norm(beta_next - beta) / max(np.linalg.norm(beta_next), 1e-12)
        if relative_change < precision:
            return beta_next, iteration, True

        beta = beta_next

    return beta, max_iter, False

In [ ]:
# Use eigenvalues to locate the useful range of step sizes.
eigenvalues = np.linalg.eigvalsh(syn_X_with_intercept.T @ syn_X_with_intercept)
stable_upper_bound = 1 / eigenvalues[-1]
theory_step = 1 / (eigenvalues[0] + eigenvalues[-1])

# Also try a small grid of step sizes and record the number of iterations.
candidate_steps = np.linspace(0.001, 0.99 * stable_upper_bound, 100)
step_results = []

for step in candidate_steps:
    beta_gd, n_iter, converged = gradient_descent(
        syn_X_with_intercept,
        syn_y,
        step_size=step,
        precision=1e-6,
    )
    if converged:
        step_results.append((step, n_iter, beta_gd))

best_step, best_iterations, best_beta = min(step_results, key=lambda row: row[1])

print('stable upper bound:', stable_upper_bound)
print('theory step:', theory_step)
print('best grid step:', best_step)
print('iterations at best grid step:', best_iterations)
print('beta from gradient descent:', best_beta)

Part (c) 的步长答案可以填

$$\boxed{0.0044}.$$

网格搜索可能给出非常接近的数，例如 `0.0042` 到 `0.0044`；题目允许 25% tolerance，所以这个量级是关键。

### Part (d-1)

mortality 数据里不同变量的尺度差别非常大，例如 `Pop` 是百万量级，`House` 只有个位数。梯度下降在这种病态尺度下会很难选步长：步长稍大就发散，步长太小又非常慢。

所以第一步应选择：

$$\boxed{\text{Scale } X \text{ and } y \text{ by the mean and standard deviation of each variable.}}$$

In [ ]:
# Load the mortality data. City is an identifier, so it is not used as a predictor.
mortality = pd.read_csv(data_dir / 'mortality.csv')

m_fields = [
    'JanTemp', 'JulyTemp', 'RelHum', 'Rain', 'Educ', 'Dens', 'NonWhite',
    'WhiteCollar', 'Pop', 'House', 'Income', 'HC', 'NOx', 'SO2'
]

X_mortality = mortality[m_fields].to_numpy(dtype=float)
y_mortality = mortality['Mortality'].to_numpy(dtype=float)

mortality[m_fields + ['Mortality']].describe().T[['mean', 'std', 'min', 'max']]

### Part (d-2)

下面把每个 predictor 和 response 都标准化成 mean 0、standard deviation 1，然后在标准化后的数据上做梯度下降。这样做以后，梯度下降得到的是 **标准化坐标系下** 的系数。

In [ ]:
def standardize_vector_or_matrix(A):
    """Return standardized data together with the mean and std used."""
    mean = A.mean(axis=0)
    std = A.std(axis=0, ddof=0)
    return (A - mean) / std, mean, std

# Standardize X and y separately, as suggested by Part (d-1).
X_mortality_scaled, X_mortality_mean, X_mortality_std = standardize_vector_or_matrix(X_mortality)
y_mortality_scaled, y_mortality_mean, y_mortality_std = standardize_vector_or_matrix(y_mortality)

# Add the intercept after standardizing the original predictors.
X_mortality_scaled_with_intercept = add_intercept(X_mortality_scaled)

# A step size just below the stability limit works well after scaling.
scaled_eigenvalues = np.linalg.eigvalsh(
    X_mortality_scaled_with_intercept.T @ X_mortality_scaled_with_intercept
)
scaled_step = 0.00419

beta_mortality_scaled_gd, mortality_iterations, mortality_converged = gradient_descent(
    X_mortality_scaled_with_intercept,
    y_mortality_scaled,
    step_size=scaled_step,
    precision=1e-10,
    max_iter=1_000_000,
)

# Compare against the closed-form solution on the same scaled data.
beta_mortality_scaled_ols = np.linalg.lstsq(
    X_mortality_scaled_with_intercept,
    y_mortality_scaled,
    rcond=None,
)[0]

print('converged:', mortality_converged)
print('iterations:', mortality_iterations)
print('scaled beta from gradient descent:')
print(beta_mortality_scaled_gd)
print('max absolute difference from scaled OLS:', np.max(np.abs(beta_mortality_scaled_gd - beta_mortality_scaled_ols)))
print('two or three significant figures:', [float(f'{b:.3g}') for b in beta_mortality_scaled_gd])

因为题目是在 Part (d-1) 标准化调整之后问 “what solution do you find”，所以答题框中通常应填标准化后的系数：

$$\boxed{[0,\ -0.234,\ -0.217,\ 0.0117,\ 0.180,\ -0.151,\ 0.109,\ 0.764,\ -0.121,\ 0.0840,\ -0.111,\ -0.0305,\ -0.996,\ 0.881,\ 0.0861]}.$$

如果想把它换回原始单位，也可以用下面的代码。换回原始单位以后会和直接对原始数据做 OLS 的解相同。

In [ ]:
# Convert standardized coefficients back to the original y and X units.
# If y_scaled = beta0 + sum_j beta_j * X_scaled_j,
# then y = y_mean + y_std * y_scaled.
original_slopes = beta_mortality_scaled_gd[1:] * y_mortality_std / X_mortality_std
original_intercept = (
    y_mortality_mean
    + y_mortality_std * beta_mortality_scaled_gd[0]
    - np.sum(original_slopes * X_mortality_mean)
)
beta_mortality_original_units = np.r_[original_intercept, original_slopes]

print('beta in original units:')
print(beta_mortality_original_units)
print('three significant figures:', [float(f'{b:.3g}') for b in beta_mortality_original_units])

### Part (e)

这里要判断哪些变量可能适合做 logarithmic transformation。一个实用判断是：如果变量明显右偏，并且取 log 后 histogram / Q-Q plot 更接近正态，那么它通常可以从 log transform 中受益。

In [ ]:
# Compare raw skewness with log-transformed skewness.
# A large reduction in absolute skewness is evidence that log transform helps.
log_candidates = ['Mortality'] + m_fields
skew_rows = []

for column in log_candidates:
    raw_values = mortality[column].to_numpy(dtype=float)
    log_values = np.log(raw_values)
    raw_skew = stats.skew(raw_values, bias=False)
    log_skew = stats.skew(log_values, bias=False)
    skew_rows.append({
        'variable': column,
        'raw_skew': raw_skew,
        'log_skew': log_skew,
        'abs_skew_reduction': abs(raw_skew) - abs(log_skew),
    })

skew_table = pd.DataFrame(skew_rows).sort_values('abs_skew_reduction', ascending=False)
skew_table

In [ ]:
# Optional visual check: raw vs log histograms and Q-Q plots.
# This cell is intentionally compact; increase figsize if you want larger plots.
fig, axes = plt.subplots(len(log_candidates), 4, figsize=(12, 2.1 * len(log_candidates)))

for row, column in enumerate(log_candidates):
    raw_values = mortality[column].to_numpy(dtype=float)
    log_values = np.log(raw_values)

    axes[row, 0].hist(raw_values, bins=12, color='steelblue', alpha=0.85)
    axes[row, 0].set_title(f'{column}: raw')
    stats.probplot(raw_values, dist='norm', plot=axes[row, 1])
    axes[row, 1].set_title('raw Q-Q')

    axes[row, 2].hist(log_values, bins=12, color='darkorange', alpha=0.85)
    axes[row, 2].set_title(f'{column}: log')
    stats.probplot(log_values, dist='norm', plot=axes[row, 3])
    axes[row, 3].set_title('log Q-Q')

    for ax in axes[row]:
        ax.tick_params(labelsize=7)

plt.tight_layout()

根据 skewness、histogram 和 Q-Q plot，我会勾选这些变量：

$$\boxed{\text{JanTemp, Dens, NonWhite, Pop, Income, HC, NOx, SO2}}.$$

`WhiteCollar` 的 log 后 skewness 也变小，但原始分布问题不算明显；如果只能提交一组，我更倾向于不勾选它。

## Problem 1.5: Report Calculations

这一题的正式回答写在报告中。这里记录关键公式和数值计算，方便检查报告中的结论。Ioannidis (2005) 的核心变量是：

- $R$: 被检验关系中，真实关系数量与非真实关系数量之比，即 pre-study odds；
- $\alpha$: Type I error rate，也就是没有真实关系时错误宣称有关系的概率；
- $\beta$: Type II error rate，因此 power 是 $1-\beta$；
- PPV: positive predictive value，即一个已经发表/宣称为真的 finding 实际为真的概率。

### Key formulas

因为 $R=\frac{\#\text{true relationships}}{\#\text{no relationships}}$，所以 pre-study probability 是

$$p=\frac{R}{R+1}.$$

没有 bias、没有多个团队抢同一个假设时，显著结果的 PPV 是

$$\mathrm{PPV}=\frac{R(1-\beta)}{R(1-\beta)+\alpha}.$$

如果有 bias，设 $u$ 是因为 bias 而被报告为 positive 的比例，则

$$\mathrm{PPV}_{bias}=\frac{uR+(1-u)R(1-\beta)}{u(1+R)+(1-u)R(1-\beta)+(1-u)\alpha}.$$

如果有 $n$ 个独立团队重复测试，且只要至少一个团队显著就算 positive claim，则

$$\mathrm{PPV}_{any}=\frac{R(1-\beta^n)}{R(1-\beta^n)+1-(1-\alpha)^n}.$$

如果科学共同体要求 $n$ 个团队全部显著，即 unanimous replication，则

$$\mathrm{PPV}_{all}=\frac{R(1-\beta)^n}{R(1-\beta)^n+\alpha^n}.$$

In [ ]:
# Problem 1.5 numerical checks

def pre_study_probability(R):
    return R / (R + 1)

def ppv_single(R, alpha, beta):
    power = 1 - beta
    return R * power / (R * power + alpha)

def rho_ratio(R, alpha, beta):
    # Ratio in Parts (5) and (6): true positives divided by
    # reported positives plus incorrectly discarded true hypotheses.
    power = 1 - beta
    return R * power / (R + alpha)

# Part (5): 20% of proposed relationships are true, so R/(R+1)=0.20.
alpha = 0.05
power = 0.70
beta = 1 - power
p_true = 0.20
R_part5 = p_true / (1 - p_true)
rho_part5 = rho_ratio(R_part5, alpha, beta)

print('Part (5) R:', R_part5)
print('Part (5) rho:', rho_part5)
print('Does testing help in Part (5)?', rho_part5 > p_true)

# Part (6): all proposed relationships are true, so take R -> infinity.
# The rho limit is simply the power, because tests keep only 70% of true hypotheses.
rho_part6 = power
print('Part (6) rho:', rho_part6)
print('Does testing help in Part (6)?', rho_part6 > 1)